In [ ]:
import os
import sys
import json
import copy
import re
from datetime import datetime

from openai import OpenAI
import yaml
import numpy as np

sys.path.append("../")
from src.utils.parser import *
from src.utils.feature_utils import theta_to_language, theta_to_state_mask

# Set your OpenAI API key
try:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
except KeyError:
    # read from ~/.openai_api_key
    with open(os.path.expanduser("~/.openai_api_key"), "r") as f:
        api_key = f.read().strip()
    client = OpenAI(api_key=api_key)

%load_ext autoreload
%autoreload 2

In [ ]:
# 19-D state layout (matches src/models/humans/frankarobot_human.py state_dim=19):
#   0-2  : end-effector x, y, z
#   3-11 : end-effector 3x3 rotation matrix, row-major
#   12-14: human x, y, z
#   15-17: laptop x, y, z
#   18   : table height
STATE_DIM = 19
STATE_DIM_LABELS = [
    "eef_x", "eef_y", "eef_z",
    "eef_rot_00", "eef_rot_01", "eef_rot_02",
    "eef_rot_10", "eef_rot_11", "eef_rot_12",
    "eef_rot_20", "eef_rot_21", "eef_rot_22",
    "human_x", "human_y", "human_z",
    "laptop_x", "laptop_y", "laptop_z",
    "table_height",
]
STATE_DIM_DESCRIPTIONS = [
    "robot end effector x position",
    "robot end effector y position",
    "robot end effector z position",
    "robot end effector rotation matrix entry (0,0)",
    "robot end effector rotation matrix entry (0,1)",
    "robot end effector rotation matrix entry (0,2)",
    "robot end effector rotation matrix entry (1,0)",
    "robot end effector rotation matrix entry (1,1)",
    "robot end effector rotation matrix entry (1,2)",
    "robot end effector rotation matrix entry (2,0)",
    "robot end effector rotation matrix entry (2,1)",
    "robot end effector rotation matrix entry (2,2)",
    "human x position",
    "human y position",
    "human z position",
    "laptop x position",
    "laptop y position",
    "laptop z position",
    "table z position (table height)",
]
assert len(STATE_DIM_LABELS) == STATE_DIM
assert len(STATE_DIM_DESCRIPTIONS) == STATE_DIM


def cleaned_output(resp, expected_len: int = STATE_DIM) -> list:
    """Extract a binary list of length `expected_len` from an LLM response."""
    raw = resp.choices[0].message.content
    lines = [ln for ln in raw.splitlines() if not ln.strip().startswith("```")]
    cleaned = "\n".join(lines).strip()

    # Prefer the last line containing an array.
    for line in reversed(cleaned.split("\n")):
        line = line.strip()
        if "[" in line and "]" in line:
            array_part = line[line.find("["): line.rfind("]") + 1]
            try:
                result = json.loads(array_part)
            except json.JSONDecodeError:
                continue
            if (
                isinstance(result, list)
                and len(result) == expected_len
                and all(x in (0, 1) for x in result)
            ):
                return result

    # Regex fallback
    pattern = r"\[\s*[01](?:\s*,\s*[01]){" + str(expected_len - 1) + r"}\s*\]"
    match = re.search(pattern, cleaned)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not find valid {expected_len}-element binary array in response: {raw!r}")


def split_trajectory(traj_states) -> str:
    """Format a trajectory as a table of the 19 state dimensions for the LLM prompt."""
    if traj_states is None or len(traj_states) == 0:
        return ""
    traj_states = np.asarray(traj_states)
    header = "Timestep | " + " | ".join(f"{n:>12s}" for n in STATE_DIM_LABELS)
    sep = "-" * len(header)
    lines = [header, sep]
    for t, state in enumerate(traj_states):
        row = " | ".join(f"{state[i]:12.3f}" for i in range(STATE_DIM))
        lines.append(f"{t:8d} | {row}")
    return "\n".join(lines)


def generate_state_mask(
    instruction: str,
    model_text: str = "gpt-4o-mini",
) -> list:
    """Map a clear instruction to a 19-D binary state mask (instruction-only)."""
    dims_text = ", ".join(f'"{d}"' for d in STATE_DIM_DESCRIPTIONS)
    messages = [
        {"role": "system", "content": (
            "You are interfacing with a robotics environment in which a robotic arm is sitting on a table. "
            "There is also a laptop on the table and a human standing next to the table. The robotic arm is "
            "learning to manipulate a cup based on some language command (e.g. 'Stay close to the table "
            "surface. Carry the cup upright.'). At each interaction, the researcher will specify the command "
            "that you need to teach the robot. In order to teach the robot, you will need to help mask the "
            f"state space by specifying what elements of the state are relevant given the command. The state "
            f"space has {STATE_DIM} elements describing the poses of the robot end effector and environment "
            f"objects: [{dims_text}]."
        )},
        {"role": "user", "content": (
            f'The language instruction for the robot is: "{instruction}"\n'
            f"Remember that the robot keeps track of {STATE_DIM} state elements: [{dims_text}].\n"
            "In order to follow the language instruction, which state elements does the robot NEED to pay "
            "attention to? Think step-by-step through each state element. For each of the "
            f"{STATE_DIM} elements, briefly explain whether the robot needs to pay attention to it for the "
            f"given instruction. Then finish with a new line that contains ONLY a JSON array of {STATE_DIM} "
            "binary values (1 = needs attention, 0 = ignore)."
        )},
    ]
    resp = client.chat.completions.create(
        model=model_text,
        messages=messages,
        temperature=0,
    )
    return cleaned_output(resp, expected_len=STATE_DIM)


def generate_state_mask_with_traj(
    instruction: str,
    traj_states,
    model_text: str = "gpt-4o-mini",
) -> list:
    """Map a clear instruction + a demonstration trajectory to a 19-D binary state mask."""
    dims_text = ", ".join(f'"{d}"' for d in STATE_DIM_DESCRIPTIONS)
    traj_text = split_trajectory(traj_states)
    messages = [
        {"role": "system", "content": (
            "You are interfacing with a robotics environment in which a robotic arm is learning to "
            "manipulate a cup based on some language command. Your role is to translate natural language "
            "instructions paired with a demonstration into a binary state-space mask for the robot. Your "
            "task is to determine which state elements need attention based on the language instruction and "
            "demonstration."
        )},
        {"role": "user", "content": (
            "CONTEXT:\n"
            "Environment: Robotic arm on a table with a laptop on the table and a human nearby.\n"
            f"State space ({STATE_DIM} dimensions): [{dims_text}].\n"
            "Task: Determine which state dimensions are relevant for following a language instruction and "
            "associated demonstration."
        )},
        {"role": "user", "content": (
            f'INSTRUCTION: "{instruction}"\n\n'
            "DEMONSTRATION DATA:\n"
            f"{traj_text}\n\n"
            "TASK: Analyze which state dimensions the robot must monitor to fulfill the instruction.\n"
            "1. First, interpret the instruction and demonstration together.\n"
            "2. For each state dimension, explain whether it needs monitoring (YES/NO) and why.\n"
            f"3. Conclude with a binary mask as a JSON array [d1, ..., d{STATE_DIM}] where 1=monitor and "
            "0=ignore.\n"
            "4. Your final line must ONLY contain this JSON array, nothing else."
        )},
    ]
    resp = client.chat.completions.create(
        model=model_text,
        messages=messages,
        temperature=0,
    )
    return cleaned_output(resp, expected_len=STATE_DIM)


def predict_state_mask(
    instruction: str,
    traj_states=None,
    model_text: str = "gpt-4o-mini",
    max_trials: int = 3,
) -> list:
    """Predict a 19-D binary mask from an instruction, optionally grounded in a trajectory."""
    for trial in range(max_trials):
        try:
            if traj_states is None:
                return generate_state_mask(instruction, model_text=model_text)
            return generate_state_mask_with_traj(instruction, traj_states, model_text=model_text)
        except Exception as e:
            print(f"State mask prediction error (trial {trial + 1}/{max_trials}): {e}")
    return None

In [ ]:
def get_theta_key(theta):
    """Generate an interpretable key for a given theta vector."""
    return '_'.join(str(x) for x in theta)


# Load configuration files
human_config = "../config/humans/frankarobot_multiple_humans.yaml"
with open(human_config, "r") as stream:
    humans_params_list = yaml.safe_load(stream)

config = "../config/reward_learning/obj20_sg10_persg5/maskedrl_llm_mask.yaml"
with open(config, "r") as stream:
    params = yaml.safe_load(stream)

# Load data split configuration
indices_file = os.path.join(params["irl"]["data_split_config_path"], "split_indices.json")
all_trajs, train_trajs, test_trajs = load_split_data(
    params["env"]["trajset_file"],
    params["env"]["per_SG"],
    params["env"]["train_test_split"],
    indices_file=indices_file,
)
print(f"Total trajectories: {len(all_trajs)} | Train: {len(train_trajs)} | Test: {len(test_trajs)}")

seed = 12345
set_seed(seed)

# Load demo indices for each theta (only needed when grounding masks in a trajectory)
demo_queries = 10
demo_indices_file = os.path.join(params["irl"]["data_split_config_path"], f"demo_indices_100_{seed}.json")
with open(demo_indices_file, "r") as f:
    saved_demo_indices = json.load(f)
    for key in saved_demo_indices.keys():
        saved_demo_indices[key] = saved_demo_indices[key][:demo_queries]
print(f"Loaded demo indices from {demo_indices_file}")

# Predict State Masks from Language Instructions

For each human, take the (clear) language instruction derived from `theta` and ask an LLM to produce a
19-D binary state mask. Set `use_trajectory = True` to also condition on a demonstration trajectory.
Accuracy is reported against the ground-truth mask from `theta_to_state_mask`.

In [ ]:
results_list = copy.deepcopy(humans_params_list["humans"])
resume_path = None
previous_resume_path = None
use_trajectory = False  # if True, also pass demo trajectory states to the LLM
model_text = "gpt-4o-mini"
accuracy_list = []

# Load resume data if specified
if resume_path is not None:
    with open(resume_path, "r") as f:
        resume_results_list = json.load(f)
    print(f"Resuming from {resume_path}")

for human_idx, human_params in enumerate(humans_params_list["humans"]):
    # Skip if already processed (when resuming)
    if resume_path is not None and "pred_state_mask" in resume_results_list[human_idx]:
        results_list[human_idx] = resume_results_list[human_idx]
        continue

    print(f"Processing human {human_idx}/{len(humans_params_list['humans'])}")

    theta = human_params["preferencer"]["theta"]
    theta_key = get_theta_key(theta)
    gt_state_mask = theta_to_state_mask(theta, state_dim=STATE_DIM).astype(int).flatten()
    language_instruction = theta_to_language([theta])[0]

    print(f"Human theta: {theta}")
    print(f"Language instruction: {language_instruction}")

    results_list[human_idx]["language_instruction"] = language_instruction
    results_list[human_idx]["theta_key"] = theta_key
    results_list[human_idx]["gt_state_mask"] = gt_state_mask

    traj_states = None
    if use_trajectory:
        if theta_key not in saved_demo_indices or not saved_demo_indices[theta_key]:
            print(f"  No demo indices for theta {theta_key}, falling back to instruction-only")
        else:
            demo_idx = saved_demo_indices[theta_key][0]
            traj_states = all_trajs[demo_idx]
            results_list[human_idx]["demo_idx"] = demo_idx

    pred = predict_state_mask(
        language_instruction,
        traj_states=traj_states,
        model_text=model_text,
    )
    if pred is None:
        print(f"  Warning: failed to predict mask for human {human_idx}")
        continue

    pred_state_mask = np.array(pred).astype(int)
    results_list[human_idx]["pred_state_mask"] = pred_state_mask

    accuracy = float(np.mean(gt_state_mask == pred_state_mask))
    accuracy_list.append(accuracy)
    print(
        f"  Avg Acc: {np.mean(accuracy_list):.3f} | Current: {accuracy:.3f} | "
        f"GT: {gt_state_mask.tolist()} | Pred: {pred_state_mask.tolist()}"
    )

    # Save checkpoint every 12 humans
    if human_idx % 12 == 0 and human_idx > 0:
        if previous_resume_path is not None:
            os.remove(previous_resume_path)
            print(f"Removed previous resume file {previous_resume_path}")

        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        suffix = "traj" if use_trajectory else "text"
        results_path = os.path.join(
            params["irl"]["data_split_config_path"],
            f"state_mask_pred_{suffix}_sdim{STATE_DIM}_{seed}_{timestamp_str}.json",
        )
        with open(results_path, "w") as f:
            json.dump(results_list, f, indent=4, default=jsonNpEncoder)
        print(f"Saved checkpoint to {results_path}")
        previous_resume_path = results_path

# Save final results
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
suffix = "traj" if use_trajectory else "text"
results_path = os.path.join(
    params["irl"]["data_split_config_path"],
    f"state_mask_pred_{suffix}_sdim{STATE_DIM}_{seed}_{timestamp_str}.json",
)
with open(results_path, "w") as f:
    json.dump(results_list, f, indent=4, default=jsonNpEncoder)
print(f"Saved final results to {results_path}")
print(f"Final average accuracy: {np.mean(accuracy_list):.3f}")